In [7]:
import re
from pathlib import Path

import pandas as pd
import plotly.express as px
from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.hf.loading import  load_tokenizer
from transformers import PreTrainedTokenizer


# EXP_DIR = get_data_dir() / 'experiments' / 'exp5.1-gsm8k-baseline'
EXP_DIR = get_data_dir() / 'experiments' / 'exp6.1-math500-baseline'

# QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'
QWEN3_4B_MODEL_ID = 'Qwen/Qwen3-4B'
# QWEN3_8B_MODEL_ID = 'Qwen/Qwen3-8B'

In [8]:
def list_exp_names() -> list[str]:
    exp_dirs = list(EXP_DIR.glob('*'))
    exp_names = [d.name for d in exp_dirs if d.is_dir()]
    return exp_names


def find_exp_dir_by_name(exp_name: str) -> Path:
    exp_dirs = list(EXP_DIR.glob(f'{exp_name}*'))
    assert len(exp_dirs) == 1, f'Expected exactly one experiment directory for name {exp_name}, found {len(exp_dirs)}'
    return exp_dirs[0]

In [9]:
# tok17b = load_tokenizer(QWEN3_1_7B_MODEL_ID)
tok4b = load_tokenizer(QWEN3_4B_MODEL_ID)
# tok8b = load_tokenizer(QWEN3_8B_MODEL_ID)

In [10]:
exp_name_and_tok_pairs = [
    # ('exp5.1.1-qwen3-1.7b-gsm8k-baseline-thinking', tok17b),
    # ('exp5.1.2-qwen3-1.7b-gsm8k-baseline-no-thinking', tok17b),
    # ('exp5.1.3-qwen3-4b-gsm8k-baseline-thinking', tok4b),
    # ('exp1.1.4-qwen3-4b-hotpotqa-baseline-no-thinking', tok4b),
    # ('exp1.1.5-qwen3-8b-hotpotqa-baseline-thinking', tok8b),
    # ('exp1.1.6-qwen3-8b-hotpotqa-baseline-no-thinking', tok8b)
    ('exp6.1.3-qwen3-4b-math500-baseline-thinking', tok4b),
    # ('exp6.1.4-qwen3-4b-math500-baseline-no-thinking', tok4b),
]

In [21]:
def get_completion_from_raw_output(raw_output: str) -> str:
    prefix_split_str = '<|im_start|>assistant\n'
    completion_with_suffix = raw_output.split(prefix_split_str)[-1].strip()
    suffix_split_str = '<|im_end|>'
    completion = completion_with_suffix.split(suffix_split_str)[0].strip()
    return completion


def get_token_len(text: str, tokenizer: PreTrainedTokenizer) -> int:
    tokens = tokenizer.encode(text)
    return len(tokens)


# def extract_answer_from_completion(completion: str) -> str:
#     completion = completion.replace('<|endoftext|>', '')

#     matches = re.findall(r"Answer: (\d+(?:\.\d+)?)<\|im_end\|>", completion)
#     if matches:
#         return matches[-1]
#     return None


def extract_answer_from_completion(completion: str) -> str:
    completion = completion.replace('<|endoftext|>', '')

    matches = re.findall(r"Answer: (.*?)<\|im_end\|>", completion, re.DOTALL)
    if matches:
        return matches[-1]
    return None

In [40]:
res_df[res_df['completion'].str.count("</think>") == 1]['completion'].apply(extract_answer_from_completion)

9      " with no extra text or explanation. In case o...
18     " with no extra text or explanation. In case o...
19     " with no extra text or explanation. In case o...
21     " with no extra text or explanation. In case o...
26     " with no extra text or explanation. In case o...
                             ...                        
481    " with no extra text or explanation. In case o...
486    " with no extra text or explanation. In case o...
490    " with no extra text or explanation. In case o...
494    " with no extra text or explanation. In case o...
497    " with no extra text or explanation. In case o...
Name: completion, Length: 76, dtype: object

In [ ]:
exp_names = list_exp_names()

accumulator = []
for exp_name, tok in exp_name_and_tok_pairs:
    exp_dir = find_exp_dir_by_name(exp_name)
    res_df = pd.read_csv(exp_dir / 'data' / 'results.csv')
    
    res_df['completion_length'] = res_df['completion'].apply(get_completion_from_raw_output).apply(lambda x: get_token_len(x, tok))
    
    if 'no-thinking' in exp_name:
        oracle_result_false_filter = res_df['oracle_result'] == False
        answer_alternative = res_df['completion'].apply(extract_answer_from_completion).str.replace(',', '').astype(float)
        answer_alternative_eq_gt = answer_alternative == res_df['ground_truth'].str.replace(',', '').astype(float)
        res_df.loc[oracle_result_false_filter, 'oracle_result'] = answer_alternative_eq_gt[oracle_result_false_filter]

    accuracy = res_df['oracle_result'].mean()

    res_df['experiment'] = exp_name
    res_df['model'] = '-'.join(exp_name.split('-')[1:3])
    res_df['thinking'] = not exp_name.endswith('no-thinking')
    
    res_df['trace_length'] = 0
    if res_df['thinking'][0]:
        res_df['trace_length'] = res_df['trace'].apply(lambda x: get_token_len(x, tok))

    accumulator.append(res_df)

    title = f'Histogram of completion lengths: thinking trace + answer<br>' \
            f'<b>Exp. name</b>: {exp_name}<br><b>Accuracy</b>: {accuracy * 100:.2f}%'
    fig = px.histogram(res_df['completion_length'],
                       nbins=50,
                       color=res_df['oracle_result'],
                       log_y=True,
                       title=title,
                       barmode='relative',
                #        color_discrete_map={True: 'blue', False: 'red'}
                       color_discrete_map={True: "#0A7BFD", False: "#FB5050"}
                       )
    fig.update_xaxes(title_text='Completion Length (tokens)')
    fig.update_yaxes(title_text='Count')
    fig.show()

df = pd.concat(accumulator, ignore_index=True)

In [ ]:
fig = px.histogram(df[(df['thinking'] == True) & (df['model'] == 'qwen3-1.7b')],
                   x='trace_length',)
fig.update_xaxes(title_text='Trace Length (tokens)')
fig.update_yaxes(title_text='Count')
fig.update_layout(title_text='Histogram of trace lengths for Qwen3-1.7B model (thinking traces only)')
fig.show() 

In [ ]:
fig = px.histogram(df[(df['thinking'] == True) & (df['model'] == 'qwen3-1.7b') & (df['trace_length'] < 50)],
                   x='trace_length',)
fig.update_xaxes(title_text='Trace Length (tokens)')
fig.update_yaxes(title_text='Count')
fig.update_layout(title_text='Histogram of trace lengths for Qwen3-1.7B model (thinking traces only)')
fig.show() 

In [ ]:
thinking_exp_name = 'exp1.1.5-qwen3-8b-hotpotqa-baseline-thinking'
no_thinking_exp_name = 'exp1.1.6-qwen3-8b-hotpotqa-baseline-no-thinking'

thinking_df = df[df['experiment'] == thinking_exp_name]
no_thinking_df = df[df['experiment'] == no_thinking_exp_name][['id', 'oracle_result']]

thinking_df = thinking_df.merge(no_thinking_df, on='id', suffixes=('_thinking', '_no_thinking'))

In [ ]:
trace_df = thinking_df[(thinking_df['oracle_result_thinking'] == True) & (thinking_df['oracle_result_no_thinking'] == False)].copy()
trace_df['trace_length'] = trace_df['trace'].apply(lambda x: get_token_len(x, tok))
trace_df = trace_df[trace_df['trace_length'] > 50].copy()

In [12]:
trace_df = trace_df[['id', 'query', 'ground_truth', 'trace']].reset_index(drop=True).copy()
# trace_df.to_csv(get_artifacts_dir() / f'exp1_1_qwen3_1_7b_traces.csv', index=False)

In [13]:
trace_df.loc[0, 'trace']

KeyError: 0